In [58]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. 데이터 생성 및 예외 데이터(노이즈) 12,000개 주입
income = np.random.randint(2000, 10000, size=2**16)
debt = np.random.randint(0, 5000, size=2**16)
target = np.where(income > debt * 1.5, 1, 0)

noise_idx = np.random.choice(2**16, size=12000, replace=False)
target[noise_idx] = 1 - target[noise_idx]

df = pd.DataFrame({'연봉': income, '대출잔액': debt, '대출승인여부': target})
X_train, X_test, y_train, y_test = train_test_split(df[['연봉', '대출잔액']], df['대출승인여부'], test_size=0.2, random_state=42)

# ---

# 2. 깊이별 결과를 저장할 빈 리스트 생성
depth_results = []

# 깊이 1부터 15까지 반복 학습
for depth in range(1, 16):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    
    train_acc = model.score(X_train, y_train) * 100
    test_acc = model.score(X_test, y_test) * 100
    
    # 결과를 딕셔너리 형태로 리스트에 담기
    depth_results.append({
        '트리 깊이(max_depth)': depth,
        '공부용 점수(Train)': round(train_acc, 2),
        '실전 점수(Test)': round(test_acc, 2)
    })

# ---

# 3. 리스트를 Pandas 표(DataFrame)로 변환하고 '트리 깊이'를 인덱스로 지정
result_df = pd.DataFrame(depth_results)
result_df.set_index('트리 깊이(max_depth)', inplace=True) # 뎁스를 인덱스 순으로 지정!

# 4. 최종 표 출력
print(result_df)

                  공부용 점수(Train)  실전 점수(Test)
트리 깊이(max_depth)                            
1                         66.53        66.68
2                         76.55        76.16
3                         78.21        77.74
4                         80.17        79.58
5                         80.89        80.32
6                         81.40        80.89
7                         81.66        80.98
8                         81.81        80.96
9                         82.04        80.80
10                        82.35        80.63
11                        82.70        80.38
12                        83.08        80.06
13                        83.53        79.91
14                        84.06        79.34
15                        84.64        78.83


In [64]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. 6만 가상 데이터 및 12,000개 노이즈 생성
income = np.random.randint(2000, 10000, size=2**16)
debt = np.random.randint(0, 5000, size=2**16)
target = np.where(income > debt * 1.5, 1, 0)

# 12,000개의 예외 데이터 심기
noise_idx = np.random.choice(2**16, size=12000, replace=False)
target[noise_idx] = 1 - target[noise_idx]

df = pd.DataFrame({'연봉': income, '대출잔액': debt, '대출승인여부': target})
X_train, X_test, y_train, y_test = train_test_split(df[['연봉', '대출잔액']], df['대출승인여부'], test_size=0.2, random_state=42)

# ---

# 2. [모델 A] 가지치기 없는 트리 (무제한 뎁스)
model_overfit = DecisionTreeClassifier(random_state=42)
model_overfit.fit(X_train, y_train)

# 3. [모델 B] 황금 뎁스로 가지치기 한 트리 (max_depth=7)
model_perfect = DecisionTreeClassifier(max_depth=7, random_state=42)
model_perfect.fit(X_train, y_train)

# ---

# 4. 최종 결과 출력
print("=== [모델 A] 가지치기 없는 트리 (과적합) ===")
print(f"트리의 최종 깊이(Depth): {model_overfit.get_depth()}")
print(f"공부용 데이터 정확도: {model_overfit.score(X_train, y_train) * 100:.1f}%")
print(f"실전(테스트) 데이터 정확도: {model_overfit.score(X_test, y_test) * 100:.1f}%")
print("-" * 40)
print("=== [모델 B] 황금 뎁스 트리 (최적화) ===")
print(f"트리의 최종 깊이(Depth): {model_perfect.get_depth()}")
print(f"공부용 데이터 정확도: {model_perfect.score(X_train, y_train) * 100:.1f}%")
print(f"실전(테스트) 데이터 정확도: {model_perfect.score(X_test, y_test) * 100:.1f}%")

=== [모델 A] 가지치기 없는 트리 (과적합) ===
트리의 최종 깊이(Depth): 73
공부용 데이터 정확도: 100.0%
실전(테스트) 데이터 정확도: 70.0%
----------------------------------------
=== [모델 B] 황금 뎁스 트리 (최적화) ===
트리의 최종 깊이(Depth): 7
공부용 데이터 정확도: 81.4%
실전(테스트) 데이터 정확도: 81.2%


In [72]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. 변수를 10개로 대폭 확장 (샘플 2**16개)
np.random.seed(100)

size = 2**16
data = {
    '연봉': np.random.randint(2000, 10000, size=size),
    '대출잔액': np.random.randint(0, 5000, size=size),
    '신용점수': np.random.randint(300, 1000, size=size),
    '나이': np.random.randint(20, 70, size=size),
    '근속연수': np.random.randint(0, 30, size=size),
    '거주지역등급': np.random.randint(1, 5, size=size),
    '기존연체횟수': np.random.randint(0, 5, size=size),
    '부양가족수': np.random.randint(0, 5, size=size),
    '연간카드사용액': np.random.randint(500, 5000, size=size),
    '자산규모': np.random.randint(1000, 50000, size=size)
}

df = pd.DataFrame(data)

# 2. 하나의 치트키 변수가 아니라, 10개 변수가 복잡하게 결합된 점수제(Score) 규칙 생성
# 변수가 많아질수록 단일 트리는 대각선이나 복잡한 차원의 경계선을 잡지 못합니다.
score = (
    (df['연봉'] * 0.3) - (df['대출잔액'] * 0.4) + (df['신용점수'] * 0.5) +
    (df['근속연수'] * 15) - (df['기존연체횟수'] * 300) + (df['자산규모'] * 0.05) -
    (df['나이'] * 2) - (df['거주지역등급'] * 50) + (df['연간카드사용액'] * 0.1)
)

# 상위 50%에게만 대출을 승인하는 가이드라인
threshold = np.median(score)
target = np.where(score > threshold, 1, 0)

# 현실적인 노이즈 20% 주입 (약 13,000개 데이터 뒤집기)
noise_idx = np.random.choice(size, size=13000, replace=False)
target[noise_idx] = 1 - target[noise_idx]
df['대출승인여부'] = target

# 데이터 분할
X = df.drop(columns=['대출승인여부'])
y = df['대출승인여부']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

# ---

# 3. 모델 정의 및 학습
# [모델 A] 단일 트리 (무제한 뎁스)
model_overfit = DecisionTreeClassifier(random_state=100)
model_overfit.fit(X_train, y_train)

# [모델 B] 단일 트리 (제한 뎁스=7)
model_pruned = DecisionTreeClassifier(max_depth=7, random_state=100)
model_pruned.fit(X_train, y_train)

# [모델 C] 랜덤 포레스트 (배깅 앙상블)
model_bagging = RandomForestClassifier(n_estimators=100, random_state=100, n_jobs=-1)
model_bagging.fit(X_train, y_train)

# ---

# 4. 결과 출력
def print_result(name, model):
    print(f"=== {name} ===")
    if hasattr(model, 'get_depth'):
        print(f"트리의 깊이(Depth): {model.get_depth()}")
    else:
        avg_depth = int(np.mean([estimator.get_depth() for estimator in model.estimators_]))
        print(f"100그루 나무의 평균 깊이(Depth): {avg_depth}")
    print(f"공부용 데이터 정확도: {model.score(X_train, y_train) * 100:.1f}%")
    print(f"실전(테스트) 데이터 정확도: {model.score(X_test, y_test) * 100:.1f}%")
    print("-" * 40)

print_result("[모델 A] 단일 트리 (무제한 뎁스)", model_overfit)
print_result("[모델 B] 단일 트리 (제한 뎁스=7)", model_pruned)
print_result("[모델 C] 랜덤 포레스트 (배깅 앙상블)", model_bagging)

=== [모델 A] 단일 트리 (무제한 뎁스) ===
트리의 깊이(Depth): 35
공부용 데이터 정확도: 100.0%
실전(테스트) 데이터 정확도: 64.5%
----------------------------------------
=== [모델 B] 단일 트리 (제한 뎁스=7) ===
트리의 깊이(Depth): 7
공부용 데이터 정확도: 75.5%
실전(테스트) 데이터 정확도: 74.4%
----------------------------------------
=== [모델 C] 랜덤 포레스트 (배깅 앙상블) ===
100그루 나무의 평균 깊이(Depth): 35
공부용 데이터 정확도: 100.0%
실전(테스트) 데이터 정확도: 77.2%
----------------------------------------


In [79]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# 1. 아까 정혁님이 돌리신 그 변수 100개짜리 똑같은 데이터셋 복사
X, y = make_classification(
    n_samples=60000, n_features=100, n_informative=80, random_state=100
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=100
)

print("🔍 1단계: 맷집 좋은 랜덤 포레스트 먼저 대충 학습 중...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=100, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_accuracy = accuracy_score(y_test, rf_model.predict(X_test))


# ========================================================
# 2. ⚡ [핵심] XGBoost의 황금 레시피를 찾기 위한 전수조사 (GridSearch)
# ========================================================
print("\n⚙️ 2단계: XGBoost 최적의 하이퍼파라미터 전수조사 시작 (시간이 조금 걸립니다)...")

# 컴퓨터가 직접 대입해볼 하이퍼파라미터 후보군 세팅
param_grid = {
    'max_depth': [6, 8],            # 나무를 아까보다 더 깊고 촘촘하게 파고들게 만듦
    'learning_rate': [0.1, 0.2],     # 보정 속도를 더 공격적으로 세팅
    'n_estimators': [100, 200]       # 나무 개수를 늘려 뒷수습 기회 증가
}

# cv=3 (3번 교차 검증)으로 툴을 돌려 억지 점수가 아닌 '진짜 실전 점수'의 1등을 찾음
grid_search = GridSearchCV(
    estimator=XGBClassifier(random_state=100, n_jobs=-1, eval_metric="logloss"),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

# 공부 시작 (이 과정에서 컴퓨터가 2 * 2 * 2 = 8가지 조합을 알아서 다 돌려봅니다)
grid_search.fit(X_train, y_train)

# 1등 먹은 최적의 모델 뽑아내기
best_xgb_model = grid_search.best_estimator_
xgb_accuracy = accuracy_score(y_test, best_xgb_model.predict(X_test))


# ==========================================
# 3. 최종 결과 복수 집행
# ==========================================
print("\n" + "=" * 60)
print(f"👑 컴퓨터가 찾아낸 XGBoost 최적 설정값: {grid_search.best_params_}")
print("=" * 60)
print(f"[배깅 계열] 랜덤 포레스트 정확도 : {rf_accuracy * 100:.2f}%")
print(f"[부스팅 계열] 튜닝된 XGBoost  정확도 : {xgb_accuracy * 100:.2f}%")
print("-" * 60)

diff = (xgb_accuracy - rf_accuracy) * 100
if diff > 0:
    print(f"🎯 성공: 대충 돌린 랜덤 포레스트보다 XGBoost가 {diff:.2f}%p 더 높게 뚫고 올라갔습니다!")
else:
    print("🚨 아직도 지다니... 후보군 범위를 더 넓혀야 합니다.")

🔍 1단계: 맷집 좋은 랜덤 포레스트 먼저 대충 학습 중...

⚙️ 2단계: XGBoost 최적의 하이퍼파라미터 전수조사 시작 (시간이 조금 걸립니다)...

👑 컴퓨터가 찾아낸 XGBoost 최적 설정값: {'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 200}
[배깅 계열] 랜덤 포레스트 정확도 : 95.85%
[부스팅 계열] 튜닝된 XGBoost  정확도 : 98.49%
------------------------------------------------------------
🎯 성공: 대충 돌린 랜덤 포레스트보다 XGBoost가 2.64%p 더 높게 뚫고 올라갔습니다!


In [80]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier  # 👈 XGBoost 라이브러리 추가

# 1. 변수를 10개로 대폭 확장 (샘플 2**16개)
np.random.seed(100)

size = 2**16
data = {
    '연봉': np.random.randint(2000, 10000, size=size),
    '대출잔액': np.random.randint(0, 5000, size=size),
    '신용점수': np.random.randint(300, 1000, size=size),
    '나이': np.random.randint(20, 70, size=size),
    '근속연수': np.random.randint(0, 30, size=size),
    '거주지역등급': np.random.randint(1, 5, size=size),
    '기존연체횟수': np.random.randint(0, 5, size=size),
    '부양가족수': np.random.randint(0, 5, size=size),
    '연간카드사용액': np.random.randint(500, 5000, size=size),
    '자산규모': np.random.randint(1000, 50000, size=size)
}

df = pd.DataFrame(data)

# 2. 10개 변수가 복잡하게 결합된 점수제(Score) 규칙 생성
score = (
    (df['연봉'] * 0.3) - (df['대출잔액'] * 0.4) + (df['신용점수'] * 0.5) +
    (df['근속연수'] * 15) - (df['기존연체횟수'] * 300) + (df['자산규모'] * 0.05) -
    (df['나이'] * 2) - (df['거주지역등급'] * 50) + (df['연간카드사용액'] * 0.1)
)

threshold = np.median(score)
target = np.where(score > threshold, 1, 0)

# 현실적인 노이즈 20% 주입 (약 13,000개 데이터 뒤집기)
noise_idx = np.random.choice(size, size=13000, replace=False)
target[noise_idx] = 1 - target[noise_idx]
df['대출승인여부'] = target

# 데이터 분할
X = df.drop(columns=['대출승인여부'])
y = df['대출승인여부']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

# ---

# 3. 모델 정의 및 학습
# [모델 A] 단일 트리 (무제한 뎁스)
model_overfit = DecisionTreeClassifier(random_state=100)
model_overfit.fit(X_train, y_train)

# [모델 B] 단일 트리 (제한 뎁스=7)
model_pruned = DecisionTreeClassifier(max_depth=7, random_state=100)
model_pruned.fit(X_train, y_train)

# [모델 C] 랜덤 포레스트 (배깅 앙상블)
model_bagging = RandomForestClassifier(n_estimators=100, random_state=100, n_jobs=-1)
model_bagging.fit(X_train, y_train)

# ⚡ [모델 D] XGBoost (기본 세팅값 - 뇌절 유발)
model_xgb_default = XGBClassifier(random_state=100, n_jobs=-1, eval_metric="logloss")
model_xgb_default.fit(X_train, y_train)

# ⚡ [모델 E] XGBoost (하이퍼파라미터 튜닝 완료 - 20% 노이즈 브레이크 장착)
model_xgb_tuned = XGBClassifier(
    n_estimators=100,      # 나무 개수 수동 지정
    max_depth=4,           # 뇌절 방지: 나무 깊이를 4층으로 딱 제한!
    learning_rate=0.05,    # 소심하고 신중하게 오답 노트를 고쳐나가라!
    random_state=100,
    n_jobs=-1,
    eval_metric="logloss"
)
model_xgb_tuned.fit(X_train, y_train)

# ---

# 4. 결과 출력
def print_result(name, model):
    print(f"=== {name} ===")
    
    # XGBoost는 트리 깊이 확인 메서드가 달라서 예외처리 해줍니다.
    if hasattr(model, 'get_depth'):
        print(f"트리의 깊이(Depth): {model.get_depth()}")
    elif hasattr(model, 'estimators_'):
        avg_depth = int(np.mean([estimator.get_depth() for estimator in model.estimators_]))
        print(f"100그루 나무의 평균 깊이(Depth): {avg_depth}")
    else:
        # XGBoost 기본 모델의 내부 booster에서 깊이 정보를 가져옵니다.
        try:
            dump = model.get_booster().get_dump()
            depths = [len(line.split(']')[0].split('-')) for line in dump if ']' in line]
            print(f"부스팅 나무들의 최대 지정 깊이(max_depth): {model.max_depth if model.max_depth else 6}")
        except:
            print("깊이 정보 출력 불가")
            
    print(f"공부용 데이터 정확도: {model.score(X_train, y_train) * 100:.1f}%")
    print(f"실전(테스트) 데이터 정확도: {model.score(X_test, y_test) * 100:.1f}%")
    print("-" * 40)

print_result("[모델 A] 단일 트리 (무제한 뎁스)", model_overfit)
print_result("[모델 B] 단일 트리 (제한 뎁스=7)", model_pruned)
print_result("[모델 C] 랜덤 포레스트 (배깅 앙상블)", model_bagging)
print_result("[모델 D] XGBoost (기본 설정값)", model_xgb_default)
print_result("[모델 E] XGBoost (튜닝 설정값)", model_xgb_tuned)

=== [모델 A] 단일 트리 (무제한 뎁스) ===
트리의 깊이(Depth): 35
공부용 데이터 정확도: 100.0%
실전(테스트) 데이터 정확도: 64.5%
----------------------------------------
=== [모델 B] 단일 트리 (제한 뎁스=7) ===
트리의 깊이(Depth): 7
공부용 데이터 정확도: 75.5%
실전(테스트) 데이터 정확도: 74.4%
----------------------------------------
=== [모델 C] 랜덤 포레스트 (배깅 앙상블) ===
100그루 나무의 평균 깊이(Depth): 35
공부용 데이터 정확도: 100.0%
실전(테스트) 데이터 정확도: 77.2%
----------------------------------------
=== [모델 D] XGBoost (기본 설정값) ===
부스팅 나무들의 최대 지정 깊이(max_depth): 6
공부용 데이터 정확도: 82.2%
실전(테스트) 데이터 정확도: 76.1%
----------------------------------------
=== [모델 E] XGBoost (튜닝 설정값) ===
부스팅 나무들의 최대 지정 깊이(max_depth): 4
공부용 데이터 정확도: 77.7%
실전(테스트) 데이터 정확도: 77.0%
----------------------------------------


In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 1. 데이터 준비 (데이터셋이 커질수록 탐색 시간이 늘어납니다)
X, y = make_classification(n_samples=5000, n_features=20, random_state=100)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

# 2. 모델 선언
xgb = XGBClassifier(random_state=100, eval_metric="logloss")

# 3. [핵심] 탐색할 파라미터 격자(Grid) 설정
# 이 안의 조합들(2*2*2 = 8가지)을 전부 다 돌려보게 됩니다.
param_grid = {
    'max_depth': [3, 5],            # 트리의 깊이
    'learning_rate': [0.01, 0.1],   # 학습률
    'n_estimators': [100, 200]      # 나무의 개수
}

# 4. GridSearchCV 설정
# cv=5는 데이터를 5등분해서 5번 교차 검증한다는 뜻입니다.
grid_search = GridSearchCV(
    estimator=xgb, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1  # CPU 코어 전부 동원 (백엔드 서버 튜닝할 때 필수!)
)

# 5. 실행 (여기서 컴퓨터가 열심히 계산합니다)
print("🔍 파라미터 최적화 탐색 중...")
grid_search.fit(X_train, y_train)

# 6. 결과 확인
print(f"최고의 점수: {grid_search.best_score_:.4f}")
print(f"최고의 조합: {grid_search.best_params_}")

# 7. 자동으로 최적의 모델을 사용하여 예측
best_model = grid_search.best_estimator_
print(f"실전 테스트 점수: {best_model.score(X_test, y_test):.4f}")